In [ ]:
# Import libraries we will use in the analysis.
# NumPy is the numerical library;
# Pandas is the library to handle data sets;
# yFinance is the library that allows us to download financial data from Yahoo;
# DateTime is the library that allows us to create variables in date format (year, month, day, minutes, seconds);
# MatPlotLib is the library to construct graphs.

import numpy as np
import pandas as pd
import yfinance as yf
import datetime as dt
import matplotlib.pyplot as plt

In [ ]:
# Set starting and ending date for dowloading data from Yahoo Finance.
# The variable "start" contains the first date we would like to download data. We choose May 1, 2020.
# The variable "end" contains the last date we would like to download data. We choose May 1, 2026.
# If you would like to use the latest available date we will use "dt.datetime.now()"

start = dt.datetime(2020,5,1)
end = dt.datetime(2026,5,1)

In [ ]:
# Download data for Bitcoin, Ether and Solana. To download data for these cryptocurrencies we need to know
# their ticker's name. For Bitcoin the ticker's name is "BTC-USD"; for Ether is "ETH-USD"; for Solana is "SOL-USD".
# To find the ticker name of a security we can google it.
# We place the downloaded data for Bitcoin in a dataframe called "btc_data";
# We place the downloaded data for Ether in a dataframe called "eth_data";
# We place the downloaded data for Solana in a dataframe called "sol_data";

btc_data = yf.download('BTC-USD', start, end, progress=False)
eth_data = yf.download('ETH-USD', start, end, progress=False)
sol_data = yf.download('SOL-USD', start, end, progress=False)

In [ ]:
# We create a new dataframe with close prices of Bitcoin, Ether and Solana.

DataSet = pd.concat([btc_data['Close'],eth_data['Close'],sol_data['Close']], axis=1)

In [ ]:
# We construct a portfolio in May 1, 2020 with 50% invested in Bitcoin, 30% in Ether, and 20% in Solana.
# We need to determine how many units of Bitcoin we can buy with 50 cents (50% of 1 dollar); How many units of Ether
# we can buy with 30 cents (30% of 1 dollar); How many units of Solana we can buy with 20 cents (20% of 1 dollar).

Quantity_BTC=0.5/DataSet.iloc[0]['BTC-USD']
Quantity_ETH=0.3/DataSet.iloc[0]['ETH-USD']
Quantity_SOL=0.2/DataSet.iloc[0]['SOL-USD']

In [ ]:
# We now compute the value of the portfolio that contains the quantities of Bitcoin, Ether and Solana
# we purchased in May 1, 2020. After the initial purchase, the quantities stay constant over time.
# However, the shares in value invested in the three cryptocurrencies change over time since the prices
# change at different rates.

DataSet['Port_Value']=DataSet['BTC-USD']*Quantity_BTC + DataSet['ETH-USD']*Quantity_ETH + DataSet['SOL-USD']*Quantity_SOL

In [ ]:
# Compute and add to DataSet daily returns for Bitcoin, Ether, Solana and the Portfolio using the
# logarithmic formula (continuous time)

DataSet['BTC_ret']=np.log(DataSet['BTC-USD'])-np.log(DataSet['BTC-USD'].shift(1))
DataSet['ETH_ret']=np.log(DataSet['ETH-USD'])-np.log(DataSet['ETH-USD'].shift(1))
DataSet['SOL_ret']=np.log(DataSet['SOL-USD'])-np.log(DataSet['SOL-USD'].shift(1))
DataSet['Port_ret']=np.log(DataSet['Port_Value'])-np.log(DataSet['Port_Value'].shift(1))

In [ ]:
# Drop rows with missing values (NaN)

DataSet=DataSet.dropna()

In [ ]:
# Compute mean and standard deviation of daily returns for Bitcoin, Ether, Solana and Portfolio.

Mean_ret=DataSet[['BTC_ret','ETH_ret','SOL_ret','Port_ret']].mean()
Var_ret=DataSet[['BTC_ret','ETH_ret','SOL_ret','Port_ret']].var()
Std_ret=DataSet[['BTC_ret','ETH_ret','SOL_ret','Port_ret']].std()
Cov_ret=DataSet[['BTC_ret','ETH_ret','SOL_ret','Port_ret']].cov()
Corr_ret=DataSet[['BTC_ret','ETH_ret','SOL_ret','Port_ret']].corr()

In [ ]:
# Print means, standard deviations and correlastions of returns

print(Mean_ret)
print()
print(Std_ret)
print()
print(Corr_ret)

In [ ]:
# So far we have considered only one portfolio allocation: initial allocation of 50% in Bitcoin, 30% in Ether, and
# 20% in Solana. In general, we may be interested in comparing several portfolios. Here we consider 41 portfolios
# characterized by different shares of Bitcoin and Ether. We create an array where each cell contains the portfolio
# share allocated to Bitcoin. The share allocated to Ether is just 1 minus the share allocated to Bitcoin.

share_vec=np.linspace(-0.2, 1.2, num=41)

# This creates an array called "share_vec" with 41 cells. The first cell is the share allocated to Bitcoin in the first
# portfolio, which is -0.2. The allocation to Ether in the first portfolio is 1-(-0.2)=1.2. This means that the
# first portfolio is "short" in Bitcoin but "long" in Ether: The investor borrows Bitcon and reinvests in Ether.
# The last portfolio allocates 1.2 to Bitcoin and -0.2 to Ether. Therefore, the portfolio is "long" in Bitcoin and
# "short" in Ether.

In [ ]:
# Now we compute the mean return and standard deviation for each of the 41 portfolios.
# We create two arrays. The first, called "mean_vec", stores the mean of returns for each portfolio.
# The second, called "std_vec", stores the standard deviations of returns for each portfolio.

mean_vec=np.zeros(len(share_vec))
std_vec=np.zeros(len(share_vec))
for i in range(len(share_vec)):
    Quantity_BTC=share_vec[i]/DataSet.iloc[0]['BTC-USD']
    Quantity_ETH=(1-share_vec[i])/DataSet.iloc[0]['ETH-USD']
    PortValue_series= DataSet['BTC-USD']*Quantity_BTC + DataSet['ETH-USD']*Quantity_ETH
    PortReturn_series=np.log(PortValue_series)-np.log(PortValue_series.shift(1))
    mean_vec[i]=PortReturn_series.mean()
    std_vec[i]=PortReturn_series.std()

In [ ]:
# Plot the standard deviation and mean return of the 41 portfolios.
# We can change default size of the plot with "plt.figure(figsize=(7, 5))"

plt.plot(std_vec, mean_vec, marker='o')
plt.xlabel('Volatility')
plt.ylabel('Expected return')
plt.show()

In [ ]:
# We now compute and plot the Sharpe Ratio for 31 portfolios. For simplicity we assume that the risk-free rate is zero.

sharpe_vec=mean_vec/std_vec

plt.plot(share_vec, sharpe_vec)
plt.xlabel('Portfolio share in Bitcoin')
plt.ylabel('Sharpe ratio')
plt.show()

In [ ]:
# Find the portfolio with the highest value of the sharpe ratio. The variable max_index will contain the position of the
# highest value.

max_index=np.argmax(sharpe_vec, axis=0)

print("Index number that maximizes Sharpe Ratio =", max_index)
print("Share of Bitcoin that maximizes Sharpe Ratio =", share_vec[max_index])
print("Value of the Sharpe Ratio with the chosen portfolio =", sharpe_vec[max_index])